In [82]:
#Import libraries
import pandas as pd
from sqlalchemy import create_engine

In [83]:
#Load the datasets
account_info = pd.read_csv('../dataset/account_info.csv')
customer_support = pd.read_csv('../dataset/customer_support.csv')
user_activity = pd.read_csv('../dataset/user_activity.csv')

In [84]:
#View the account_info df
account_info.head()

,customer_id,email,state,plan,plan_list_price,churn_status
0,C10000,user10000@example.com,New Jersey,Enterprise,105,Y
1,C10001,user10001@example.net,Louisiana,Basic,22,Y
2,C10002,user10002@example.net,Oklahoma,Basic,24,NaN
3,C10003,user10003@example.com,Michigan,Free,0,NaN
4,C10004,user10004@example.com,Texas,Enterprise,119,NaN


In [85]:
#View the customer_support df
customer_support.head()

,ticket_time,user_id,channel,topic,resolution_time_hours,state,comments
0,2025-06-13 05:55:17.154573,10125,chat,technical,11.48,1,NaN
1,2025-08-06 13:21:54.539551,10109,chat,account,1.01,0,NaN
2,2025-08-22 12:39:35.718663,10149,chat,technical,10.09,0,Erase my data from your systems.
3,2025-06-07 02:49:46.986055,10268,phone,account,9.10,1,NaN
4,2025-07-25 00:24:38.945079,10041,phone,other,2.28,1,NaN


In [86]:
#View user_activity df
user_activity.head()

,event_time,user_id,event_type
0,2025-09-08 15:05:39.422721,10118,watch_video
1,2025-09-08 08:15:05.264103,10220,watch_video
2,2025-11-14 06:28:35.207671,10009,share_workout
3,2025-08-20 16:53:38.682901,10227,read_article
4,2025-07-24 16:47:31.728422,10123,track_workout


CLEANING OF ACCOUNT_INFO TABLE

In [87]:
#View the shape
account_info.shape

(400, 6)

In [88]:
#View the info
account_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   customer_id      400 non-null    object
 1   email            400 non-null    object
 2   state            400 non-null    object
 3   plan             400 non-null    object
 4   plan_list_price  400 non-null    int64 
 5   churn_status     114 non-null    object
dtypes: int64(1), object(5)
memory usage: 18.9+ KB


In [89]:
#View the summary
account_info.describe(include='all')

,customer_id,email,state,plan,plan_list_price,churn_status
count,400,400,400,400,400.000000,114
unique,400,400,50,4,NaN,1
top,C10000,user10000@example.com,Virginia,Basic,NaN,Y
freq,1,1,16,118,NaN,114
mean,NaN,NaN,NaN,NaN,43.965000,NaN
std,NaN,NaN,NaN,NaN,45.263348,NaN
min,NaN,NaN,NaN,NaN,0.000000,NaN
25%,NaN,NaN,NaN,NaN,0.000000,NaN
50%,NaN,NaN,NaN,NaN,26.000000,NaN
75%,NaN,NaN,NaN,NaN,77.250000,NaN


In [90]:
#Review null values
account_info.isna().sum()

customer_id          0
email                0
state                0
plan                 0
plan_list_price      0
churn_status       286
dtype: int64

In [91]:
#Verify duplicates
account_info.duplicated().sum()

0

In [92]:
#======customer_id column validation=======
#the column name does not match with the key column of two other tables
#standarize the column name
account_info = account_info.rename(columns={'customer_id':'user_id'})

#review columns name
account_info.columns

Index(['user_id', 'email', 'state', 'plan', 'plan_list_price', 'churn_status'], dtype='object')

In [93]:
#the key values have a prefix 'C', while the other two key columns are just number
#Standarize the key columns
#Remove 'C'
account_info['user_id'] = account_info['user_id'].str.replace('C', '')

#trasnform into int data type
account_info['user_id'] = account_info['user_id'].astype('int')

#view the first five rows
account_info['user_id'].head()

0    10000
1    10001
2    10002
3    10003
4    10004
Name: user_id, dtype: int32

In [94]:
#Validate the uniqueness of key values
account_info['user_id'].nunique()

400

In [95]:
#=========email column validation========
#Validate the uniqueness of emails
account_info['email'].nunique()

400

In [96]:
#=========state column validation========
#Unique values, there should be 50 states
print(f'There are {account_info['state'].nunique()} unique states')

print(account_info['state'].unique())

There are 50 unique states
['New Jersey' 'Louisiana' 'Oklahoma' 'Michigan' 'Texas' 'Ohio' 'Kentucky'
 'North Carolina' 'Alaska' 'Virginia' 'Kansas' 'Arizona' 'Idaho'
 'Connecticut' 'Colorado' 'Rhode Island' 'Maine' 'Wisconsin' 'Vermont'
 'Florida' 'Delaware' 'New York' 'Montana' 'Missouri' 'West Virginia'
 'New Mexico' 'Iowa' 'North Dakota' 'Hawaii' 'Nebraska' 'South Dakota'
 'Utah' 'Washington' 'Pennsylvania' 'Nevada' 'Arkansas' 'Illinois'
 'Wyoming' 'California' 'Tennessee' 'Minnesota' 'Maryland' 'Massachusetts'
 'Mississippi' 'South Carolina' 'Oregon' 'Georgia' 'Indiana' 'Alabama'
 'New Hampshire']


In [97]:
#=========plan column validation===========
#unique values
print(account_info['plan'].unique())
print(account_info['plan'].value_counts())

['Enterprise' 'Basic' 'Free' 'Pro']
plan
Basic         118
Free          105
Enterprise     92
Pro            85
Name: count, dtype: int64


In [98]:
#========plan_list_price column validation ===========
#summary
print(account_info['plan_list_price'].describe())

#Cros validate of price by plan
print(account_info.groupby('plan')['plan_list_price'].agg(['min', 'max', 'mean']))

count    400.000000
mean      43.965000
std       45.263348
min        0.000000
25%        0.000000
50%       26.000000
75%       77.250000
max      148.000000
Name: plan_list_price, dtype: float64
            min  max        mean
plan                            
Basic        10   30   19.872881
Enterprise   81  148  116.010870
Free          0    0    0.000000
Pro          32   80   53.741176


In [99]:
#==========churn_status column validation ==========
#unique values
print(account_info['churn_status'].unique())

#All null values is considered as no churn
#Replace all null values by 'No' label
account_info['churn_status'] = account_info['churn_status'].fillna('No')

#Verify the unique values
print(account_info['churn_status'].unique())

#'Y' label is not understandable, so change by 'Yes'
account_info['churn_status'] = account_info['churn_status'].str.replace('Y', 'Yes')

#Verfiy
print(account_info['churn_status'].unique())

['Y' nan]
['Y' 'No']
['Yes' 'No']


CLEANING OF CUSTOMER_SUPPORT TABLE

In [100]:
#View the shape
customer_support.shape

(918, 7)

In [101]:
#View the info
customer_support.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ticket_time            918 non-null    object 
 1   user_id                918 non-null    int64  
 2   channel                918 non-null    object 
 3   topic                  918 non-null    object 
 4   resolution_time_hours  918 non-null    float64
 5   state                  918 non-null    int64  
 6   comments               46 non-null     object 
dtypes: float64(1), int64(2), object(4)
memory usage: 50.3+ KB


In [102]:
#Summary
customer_support.describe(include='all')

,ticket_time,user_id,channel,topic,resolution_time_hours,state,comments
count,918,918.000000,918,918,918.000000,918.000000,46
unique,918,NaN,4,4,NaN,NaN,19
top,2025-06-13 05:55:17.154573,NaN,email,billing,NaN,NaN,Wipe all data you hold.
freq,1,NaN,298,239,NaN,NaN,5
mean,NaN,10201.985839,NaN,NaN,10.391362,0.549020,NaN
std,NaN,116.048475,NaN,NaN,7.079888,0.497863,NaN
min,NaN,10000.000000,NaN,NaN,0.520000,0.000000,NaN
25%,NaN,10103.000000,NaN,NaN,5.112500,0.000000,NaN
50%,NaN,10205.500000,NaN,NaN,9.040000,1.000000,NaN
75%,NaN,10302.750000,NaN,NaN,13.137500,1.000000,NaN


In [103]:
#Cheack duplicates
customer_support.duplicated().sum()

0

In [104]:
#Cheack missing values
customer_support.isna().sum()

ticket_time                0
user_id                    0
channel                    0
topic                      0
resolution_time_hours      0
state                      0
comments                 872
dtype: int64

In [105]:
#============ticket_time column validation==============
#Transform into datetime data type
customer_support['ticket_time'] = pd.to_datetime(customer_support['ticket_time'])

#cheack null values
print(f'There are {customer_support['ticket_time'].isnull().sum()} null values')

#Ckeck min and max dates
print(f'Min date: {customer_support['ticket_time'].min()}')
print(f'Max date: {customer_support['ticket_time'].max()}')

There are 0 null values
Min date: 2025-06-05 15:32:33.005817
Max date: 2025-12-01 22:01:58.485299


In [106]:
#==========user_id column validation===============
#This is foreing key column, so it is expected that there are duplicated keys
print(f'{customer_support['user_id'].nunique()} unique values')
print(f'{customer_support['user_id'].duplicated().sum()} duplicates')

#Check min and max values
print(f'Min value: {customer_support['user_id'].min()}')
print(f'Max value: {customer_support['user_id'].max()}')
#all the foreing keys are inside the range of primary key values

367 unique values
551 duplicates
Min value: 10000
Max value: 10399


In [107]:
#============channel column validation============
#check unique values
print(customer_support['channel'].unique())
print(customer_support['channel'].value_counts())

#There are 39 missing values '-' (unclear)
#Replace '-' by 'unknown' (clear)
customer_support['channel'] = customer_support['channel'].str.replace('-', 'unknown')
print(customer_support['channel'].value_counts())

['chat' 'phone' '-' 'email']
channel
email    298
chat     294
phone    287
-         39
Name: count, dtype: int64
channel
email      298
chat       294
phone      287
unknown     39
Name: count, dtype: int64


In [108]:
#================topic column validation===============
#check unique values
print(customer_support['topic'].unique())#there are four topics

['technical' 'account' 'other' 'billing']


In [109]:
#==============Resolution_time_hour column validation===============
#Summary
print(customer_support['resolution_time_hours'].describe())
#Check negative values
print(f'Number of negative values: {customer_support[customer_support['resolution_time_hours']<0]['resolution_time_hours'].count()}')

count    918.000000
mean      10.391362
std        7.079888
min        0.520000
25%        5.112500
50%        9.040000
75%       13.137500
max       32.460000
Name: resolution_time_hours, dtype: float64
Number of negative values: 0


In [110]:
#============state column validation=============
#check unique values
print(customer_support['state'].unique())#Is a binary column
#Create a new column to label state column (unclear)
label = {1: 'Solved', 0: 'Open'}
customer_support['state_label'] = customer_support['state'].map(label)

#check the new column
print(customer_support[['state', 'state_label']])
print(customer_support['state_label'].unique())

[1 0]
     state state_label
0        1      Solved
1        0        Open
2        0        Open
3        1      Solved
4        1      Solved
..     ...         ...
913      0        Open
914      0        Open
915      0        Open
916      1      Solved
917      0        Open

[918 rows x 2 columns]
['Solved' 'Open']


In [111]:
#==========comments column validation=========
#check null values
print(customer_support['comments'].isna().sum())

#Replace null values by 'unknown'
customer_support['comments'] = customer_support['comments'].fillna('unknown')

#check null values
print(customer_support['comments'].isna().sum())


872
0


USER_ACTIVITY TABLE

In [112]:
#View table shape
user_activity.shape

(445, 3)

In [113]:
#View info
user_activity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 445 entries, 0 to 444
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   event_time  445 non-null    object
 1   user_id     445 non-null    int64 
 2   event_type  445 non-null    object
dtypes: int64(1), object(2)
memory usage: 10.6+ KB


In [114]:
#Summary
user_activity.describe(include='all')

,event_time,user_id,event_type
count,445,445.000000,445
unique,445,NaN,4
top,2025-09-08 15:05:39.422721,NaN,read_article
freq,1,NaN,125
mean,NaN,10198.930337,NaN
std,NaN,113.583807,NaN
min,NaN,10000.000000,NaN
25%,NaN,10110.000000,NaN
50%,NaN,10197.000000,NaN
75%,NaN,10294.000000,NaN


In [115]:
#Check duplicates
print(user_activity.duplicated().sum())

0


In [116]:
#Check null values
print(user_activity.isna().sum())

event_time    0
user_id       0
event_type    0
dtype: int64


In [117]:
#==========user_time column validation========
#transform into datetime data type
user_activity['event_time'] = pd.to_datetime(user_activity['event_time'])

#check min and max dates
print(f'Min date: {user_activity['event_time'].min()}')
print(f'Max date: {user_activity['event_time'].max()}')

Min date: 2025-06-05 10:14:53.039663
Max date: 2025-12-01 21:12:13.342817


In [118]:
#==============event_type column validation============
#check unique values
print(user_activity['event_type'].unique())

#replace the undescore by space
user_activity['event_type'] = user_activity['event_type'].str.replace('_', ' ')

#check again
print(user_activity['event_type'].unique())

['watch_video' 'share_workout' 'read_article' 'track_workout']
['watch video' 'share workout' 'read article' 'track workout']


LOAD THE CLEAN DATASET TO POSTGRESQL DATABASE

In [119]:
#Step 1: connect to PostgreSQL
#replace placeholders with current details
username = 'postgres'
password = 'sql2026'
host = 'localhost'
port = '5432'
database = 'churn_analysis'

engine = create_engine(f'postgresql://{username}:{password}@{host}:{port}/{database}')

#Step 2: load the dataframe into PostgreSQL
table_1 ='account_info'
account_info.to_sql(table_1, engine, if_exists='replace', index=False)
print(f'Data successfully loaded into table {table_1} in database {database}.')

table_2 = 'customer_support'
customer_support.to_sql(table_2, engine, if_exists='replace', index=False)
print(f'Data successfully loaded into table {table_2} in database {database}.')

table_3 = 'user_activity'
user_activity.to_sql(table_3, engine, if_exists='replace', index=False)
print(f'Data successfully loaded into table {table_3} in database {database}.')


Data successfully loaded into table account_info in database churn_analysis.
Data successfully loaded into table customer_support in database churn_analysis.
Data successfully loaded into table user_activity in database churn_analysis.
